# Reprodutibilidade e diagnóstico das dependências do PySITS no WSL2

## 1. Objetivo

O notebook tem como objetivo a reprodução da instalação do PySITS no WSL2, tomando como referência o repositório original `mlops-lulc`.

Assim, vamos identificar problemas de compatibilidade entre dependências, registrar erros encontrados durante a instalação e execução e documentar as modificações necessárias para viabilizar o funcionamento do PySITS.

## 2. Ambiente experimental

O experimento é conduzido usando WSL2, usando o VS Code como interface de desenvolvimento.


## 3. Metodologia

A investigação segue os passos:

1. Caracterização do ambiente computacional disponível.
2. Reprodução do procedimento original de instalação.
3. Identificação e registro de erros e incompatibilidades.

## 4. Configuração original do ambiente

Registramos as versões das dependências principais envolvendo a instalação do PySITS.

- Python: 3.10.22 | packaged by conda-forge | (main, Oct  6 2026, 23:02:24) [GCC 14.4.0]
- pysits: 1.5.4
- pandas: 2.3.3
- rpy2: 3.6.9
- pyarrow: 21.0.0

No terminal `R --version`
- R: 4.4.3 (2025-02-28) -- "Trophy Case"

In [1]:
import sys
import importlib.metadata as metadata

print("Python:", sys.version)

packages = ["pysits", "pandas", "rpy2", "pyarrow"]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.10.22 | packaged by conda-forge | (main, Oct  6 2026, 23:02:24) [GCC 14.4.0]
pysits: 1.5.4
pandas: 2.3.3
rpy2: 3.6.9
pyarrow: 21.0.0


## 5. Identificação dos erros de instalação
### 5.1. Erro inicial de importação

Depois de instalar o PySITS e suas dependências, tentamos a importação do pacote para verificar a instalação.

In [2]:
from pysits import *
print("PySITS OK")

Error importing in API mode: ModuleNotFoundError("No module named '_rinterface_cffi_api'")
Trying to import in ABI mode.


error: symbol 'R_getVar' not found in library '/home/sofia/miniforge3/envs/mlops_lulc/lib/R/lib/libR.so': /home/sofia/miniforge3/envs/mlops_lulc/lib/R/lib/libR.so: undefined symbol: R_getVar

### 5.2. Análise do erro

A falha mostrou duas mensagens relevantes:

1. Error importing in API mode: 
    ModuleNotFoundError("No module named `_rinterface_cffi_api`")
--> esse erro foi encontrado no Geolab e não impediu a importação.

2. error: symbol `R_getVar` not found in library
`/home/sofia/miniforge3/envs/mlops_lulc/lib/R/lib/libR.so`
--> Essa falha impediu a continuidade da importação. O rpy2 tentou acessar o símbolo R_getVar na biblioteca compartilhada do R (libR.so), mas não conseguiu localizá-lo.

A falha ocorreu durante a inicialização da interface entre Python e R, por meio do rpy2 3.6.9. 

A mensagem undefined symbol: R_getVar indica que o símbolo solicitado pelo rpy2 não foi encontrado na biblioteca compartilhada do R (libR.so).

Isso sugere uma possível incompatibilidade entre a versão ou compilação do R (4.4.3) e a interface utilizada pelo rpy2. Mas, não se pode afirmar que a falha se deu exclusivamente pela versão do R sem testes adicionais de compatibilidade.

**Resultado:** Falha na integração entre Python e R durante a importação do PySITS.

## 6. Considerações finais

A reprodução do ambiente original do repositório `mlops-lulc` no WSL2 permitiu identificar uma falha na integração entre Python e R durante a importação do PySITS. Embora a instalação do pacote e de suas dependências tenha sido concluída, a execução foi interrompida devido à ausência do símbolo `R_getVar` na biblioteca compartilhada do R (`libR.so`), utilizado pelo `rpy2` durante a inicialização da interface entre as linguagens.

O erro indica uma possível incompatibilidade entre as versões ou compilações do R e do `rpy2` presentes no ambiente experimental. Entretanto, como não foram realizados testes isolando essas dependências, não é possível atribuir a causa da falha exclusivamente à versão do R.

Dessa forma, o experimento evidencia que a reprodução do ambiente original não garantiu o funcionamento do PySITS, apesar da instalação bem-sucedida de suas dependências. Os resultados também reforçam a importância de documentar não apenas as versões dos pacotes utilizados, mas também as condições de integração entre Python e R, especialmente por se tratar de uma dependência fundamental para o funcionamento do PySITS.